In [4]:
import getpass

import os



import pandas as pd

import requests



BASE_URL = "https://v3.football.api-sports.io"

API_KEY = os.getenv("API_FOOTBALL_KEY")

LIGUE_ID = 61  # Ligue 1

SAISON = 2023

NOMBRE_JOUEURS = 20

In [3]:
def recuperer_joueurs(api_key, ligue_id, saison, nombre_joueurs=20):

    """Récupère des joueurs API-Football avec leur identifiant."""

    if not api_key:

        raise ValueError("Une clé API-Football est requise.")

    if nombre_joueurs <= 0:

        return pd.DataFrame()



    session = requests.Session()

    session.headers.update({"x-apisports-key": api_key})

    joueurs = []

    page = 1



    while len(joueurs) < nombre_joueurs:

        reponse = session.get(

            f"{BASE_URL}/players",

            params={"league": ligue_id, "season": saison, "page": page},

            timeout=30,

        )

        reponse.raise_for_status()

        resultat = reponse.json()



        if resultat.get("errors"):

            raise RuntimeError(f"Erreur API-Football : {resultat['errors']}")



        elements = resultat.get("response", [])

        joueurs.extend(element.get("player", {}) for element in elements)



        pagination = resultat.get("paging", {})

        if not elements or page >= pagination.get("total", page):

            break

        page += 1



    colonnes = ["id", "name", "firstname", "lastname", "age", "nationality"]

    donnees = pd.json_normalize(joueurs[:nombre_joueurs])

    return donnees.reindex(columns=colonnes)

In [ ]:
cle_api = API_KEY or getpass.getpass("Clé API-Football : ")



joueurs = recuperer_joueurs(

    cle_api,

    ligue_id=LIGUE_ID,

    saison=SAISON,

    nombre_joueurs=NOMBRE_JOUEURS,

)

joueurs